# YOLO12-Small + GhostConv + EMA — CH-RDD2022 (Kaggle)

Notebook ini memakai **repository yang di-clone** sebagai sumber implementasi `GhostConv`, `EMAAttention`, dan YAML model. EMA di sini berarti *Efficient Multi-scale Attention*, bukan *Exponential Moving Average* untuk bobot training.

Arsitektur: Conv standar + EMA pada P3/8 untuk detail cacat kecil, serta GhostConv pada downsampling P4/16 dan P5/32 untuk efisiensi. Hasil training, evaluasi validation/test, konfigurasi, dan source modifikasi dikemas sebagai ZIP di `/kaggle/working`.

Sebelum menjalankan, pilih **Accelerator: GPU** dan aktifkan Internet apabila Kaggle belum memiliki dependency yang diperlukan.

In [ ]:
# 1. Clone branch modifikasi dan install repository secara editable.
import json
import platform
import re
import subprocess
import sys
import zipfile
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
REPO_BRANCH = 'yolo12-ghost-ema'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022'

def log_section(title: str) -> None:
    print(f'\n{"=" * 90}\n{title}\n{"=" * 90}')

log_section('CLONE AND INSTALL MODIFIED REPOSITORY')
if REPO_DIR.exists():
    print(f'Repository sudah ada. Menyinkronkan branch {REPO_BRANCH} ...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--force', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
REPO_METADATA = WORKDIR / 'repository_revision.txt'
REPO_METADATA.write_text(
    f'repository={REPO_URL}\nbranch={REPO_BRANCH}\ncommit={REPO_COMMIT}\n', encoding='utf-8'
)

sys.path.insert(0, str(REPO_DIR))
import torch
import ultralytics

log_section('KAGGLE ENVIRONMENT')
print(f'Python          : {platform.python_version()}')
print(f'PyTorch         : {torch.__version__}')
print(f'Ultralytics     : {ultralytics.__version__}')
print(f'Ultralytics path: {Path(ultralytics.__file__).resolve()}')
print(f'CUDA ready      : {torch.cuda.is_available()}')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU tidak tersedia; training pada CPU akan sangat lambat.')
print(f'Repository commit: {REPO_COMMIT}')

In [ ]:
# 2. Dataset dan hyperparameter. Ubah DATA_ROOT hanya jika nama Kaggle Dataset Anda berbeda.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd2022/datasets-china-split')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12-ghost-ema.yaml'
CUSTOM_SOURCE_FILES = (
    REPO_DIR / 'ultralytics/nn/modules/conv.py',
    REPO_DIR / 'ultralytics/nn/modules/__init__.py',
    REPO_DIR / 'ultralytics/nn/tasks.py',
    MODEL_YAML,
)

# Hyperparameter pelatihan yang sama dengan eksperimen GhostConv+ECA sebelumnya.
EPOCHS = 160
IMGSZ = 640
BATCH = 64
OPTIMIZER = 'SGD'
LR0 = 0.01
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005
PATIENCE = 0
WORKERS = 2
SEED = 42
EXPERIMENT_NAME = 'yolo12s_ghost_ema_ch_rdd2022_pretrained'
RUNS_DIR = WORKDIR / 'runs'

DATA_YAML.write_text(
    f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''',
    encoding='utf-8',
)

log_section('DATASET AND TRAINING CONFIGURATION')
print(DATA_YAML.read_text(encoding='utf-8'))
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split in ('train', 'val', 'test'):
    image_dir, label_dir = DATA_ROOT / split / 'images', DATA_ROOT / split / 'labels'
    image_count = sum(p.suffix.lower() in image_extensions for p in image_dir.rglob('*')) if image_dir.exists() else 0
    label_count = len(list(label_dir.glob('*.txt'))) if label_dir.exists() else 0
    print(f'{split:>5}: {image_count:>6} images | {label_count:>6} labels')
    if split in {'train', 'val'}:
        assert image_count > 0, f'Tidak ada gambar pada {image_dir}'

print(f'Hyperparameter: epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, optimizer={OPTIMIZER}, lr0={LR0}')
print('Catatan: batch 64 mengikuti eksperimen sebelumnya. Jika OOM, turunkan batch dan catat perubahan tersebut.')

In [ ]:
# 3. Verifikasi modul dari repository hasil clone dan cetak model info.
from ultralytics.nn.modules import EMAAttention, GhostConv
from ultralytics.nn.tasks import DetectionModel

assert MODEL_YAML.exists(), f'Model YAML tidak ditemukan: {MODEL_YAML}'
assert all(path.exists() for path in CUSTOM_SOURCE_FILES), 'File implementasi Ghost-EMA tidak lengkap.'

log_section('YOLO12S + GHOSTCONV + EMA IMPLEMENTATION')
print(f'EMA module : {EMAAttention.__module__}.{EMAAttention.__name__}')
print(f'Ghost module: {GhostConv.__module__}.{GhostConv.__name__}')
print(f'Model YAML : {MODEL_YAML}')
print(MODEL_YAML.read_text(encoding='utf-8'))

check_model = DetectionModel(str(MODEL_YAML), nc=5, verbose=False)
parameter_count = sum(parameter.numel() for parameter in check_model.parameters())
ema_layers = [(layer.i, layer.groups) for layer in check_model.model if isinstance(layer, EMAAttention)]
ghost_layers = [layer.i for layer in check_model.model if isinstance(layer, GhostConv)]
print(f'Parameters (5 classes): {parameter_count:,}')
print(f'EMA layers (index, groups): {ema_layers}')
print(f'GhostConv layer indices : {ghost_layers}')
print('Expected layout        : EMA(P3, factor=8), GhostConv(P4, P5)')
check_model.info(detailed=False, verbose=True)
del check_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 4. Transfer bobot compatible dari YOLO12s pretrained. EMA dan GhostConv dipelajari saat fine-tuning.
from ultralytics import YOLO

PRETRAINED_WEIGHTS = 'yolo12s.pt'

def target_layer_index(source_index: int) -> int:
    """Map index layer source setelah EMA ditambahkan setelah layer backbone 4."""
    return source_index + int(source_index >= 5)

def remap_yolo12s_weights(source_state: dict, target_state: dict) -> dict:
    """Transfer tensor hanya bila nama hasil remap dan bentuknya cocok."""
    transferred = {}
    pattern = re.compile(r'^model\.(\d+)(\..+)$')
    for source_key, source_tensor in source_state.items():
        match = pattern.match(source_key)
        if match is None:
            continue
        target_key = f'model.{target_layer_index(int(match.group(1)))}{match.group(2)}'
        if target_key in target_state and target_state[target_key].shape == source_tensor.shape:
            transferred[target_key] = source_tensor
    return transferred

log_section('PRETRAINED WEIGHT TRANSFER')
model = YOLO(str(MODEL_YAML))
source_model = YOLO(PRETRAINED_WEIGHTS).model.float()
target_state = model.model.state_dict()
transferred_state = remap_yolo12s_weights(source_model.state_dict(), target_state)
incompatible = model.model.load_state_dict(transferred_state, strict=False)
PRETRAINED_REPORT = {
    'source_weights': PRETRAINED_WEIGHTS,
    'transferred_tensors': len(transferred_state),
    'target_tensors': len(target_state),
    'uninitialized_tensors': len(incompatible.missing_keys),
    'uninitialized_tensor_names': incompatible.missing_keys,
}
# Trainer memakai checkpoint ini untuk menjaga tensor pretrained yang kompatibel setelah nc diubah menjadi 5.
model.ckpt = {'model': model.model}
del source_model, target_state, transferred_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Transferred tensors : {PRETRAINED_REPORT['transferred_tensors']}/{PRETRAINED_REPORT['target_tensors']}")
print(f"Uninitialized tensors: {PRETRAINED_REPORT['uninitialized_tensors']} (EMA/GhostConv dan head kelas)")
print('Training akan membangun head 5 kelas dan mempertahankan tensor pretrained yang kompatibel.')

In [ ]:
# 5. Training — log per epoch dan grafik akan terlihat pada output Kaggle.
log_section('TRAINING STARTED')
print(f'Experiment : {EXPERIMENT_NAME}')
print(f'Device     : {DEVICE}')
print(f'Epochs     : {EPOCHS} | imgsz: {IMGSZ} | batch: {BATCH}')
print(f'Optimizer  : {OPTIMIZER} | lr0: {LR0} | momentum: {MOMENTUM} | weight_decay: {WEIGHT_DECAY}')

model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name=EXPERIMENT_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer=OPTIMIZER,
    lr0=LR0,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    cos_lr=False,
    patience=PATIENCE,
    seed=SEED,
    plots=True,
    verbose=True,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = Path(model.trainer.best)
LAST_PT = Path(model.trainer.last)
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')
print(f'Last weights : {LAST_PT}')

In [ ]:
# 6. Evaluasi best checkpoint dan simpan angka validation/test secara eksplisit.
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))

def metric_summary(metrics) -> dict:
    return {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'save_dir': str(metrics.save_dir),
    }

val_metrics = best_model.val(
    data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=16, device=DEVICE,
    project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_val', exist_ok=True, plots=True,
)
EVALUATION_REPORT = {'validation': metric_summary(val_metrics)}
print(json.dumps(EVALUATION_REPORT['validation'], indent=2))

test_label_dir = DATA_ROOT / 'test' / 'labels'
if test_label_dir.exists() and any(test_label_dir.glob('*.txt')):
    test_metrics = best_model.val(
        data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=16, device=DEVICE,
        project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_test', exist_ok=True, plots=True,
    )
    EVALUATION_REPORT['test'] = metric_summary(test_metrics)
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
    print(json.dumps(EVALUATION_REPORT['test'], indent=2))
else:
    prediction_results = best_model.predict(
        source=str(DATA_ROOT / 'test' / 'images'), imgsz=IMGSZ, device=DEVICE, conf=0.25,
        save=True, save_txt=True, project=str(RUNS_DIR),
        name=f'{EXPERIMENT_NAME}_test_predictions', exist_ok=True, verbose=True,
    )
    TEST_OUTPUT_DIR = Path(prediction_results[0].save_dir) if prediction_results else RUNS_DIR
    EVALUATION_REPORT['test'] = {'status': 'labels unavailable; prediction only', 'save_dir': str(TEST_OUTPUT_DIR)}

EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
print(f'Evaluation metrics saved: {EVALUATION_JSON}')

In [ ]:
# 7. Arsipkan checkpoint, hasil, konfigurasi, metrik, dan kode modifikasi ke ZIP.
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(
    json.dumps(
        {
            'dataset_root': str(DATA_ROOT),
            'repository_url': REPO_URL,
            'repository_branch': REPO_BRANCH,
            'repository_commit': REPO_COMMIT,
            'pretrained_transfer': PRETRAINED_REPORT,
            'ema_position': 'P3/8 after backbone layer 4; factor=8',
            'ghost_conv_positions': 'P4/16 and P5/32 downsampling',
            'model_yaml': str(MODEL_YAML),
            'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH,
            'optimizer': OPTIMIZER, 'lr0': LR0, 'momentum': MOMENTUM,
            'weight_decay': WEIGHT_DECAY, 'patience': PATIENCE, 'seed': SEED,
            'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
        },
        indent=2,
    ),
    encoding='utf-8',
)
ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'

def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

log_section('CREATE RESULTS ZIP')
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    count = 0
    for artifact in (RUN_DIR, Path(val_metrics.save_dir), TEST_OUTPUT_DIR, DATA_YAML, *CUSTOM_SOURCE_FILES, REPO_METADATA, RUN_CONFIG, EVALUATION_JSON):
        count += add_to_zip(archive, Path(artifact))

print(f'ZIP created : {ZIP_PATH}')
print(f'ZIP size    : {ZIP_PATH.stat().st_size / (1024 ** 2):.2f} MB')
print(f'Files added : {count}')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))